# 06 — Level 2 source-specific analysis
Reports support-threshold retention for 5, 10, and 20 using training support only. The training-derived eligible label set is applied unchanged to validation and test. PwC tasks and EDAM Topics remain separate label spaces.


In [ ]:
from pathlib import Path
import json, csv, subprocess, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass
REPO=Path('/content/research_software_classification_attributes')
if not REPO.exists():
    subprocess.run(['git','clone','-b','data-finalization','https://github.com/kuefmz/research_software_classification_attributes.git',str(REPO)],check=True)
sys.path.insert(0,str(REPO))
ROOT=Path('/content/drive/MyDrive/phd_research_software')
DATA=ROOT/'data/frozen/level2_fine_grained.jsonl'
SPLIT=ROOT/'splits/level2_fine_grained_group_aware_seed42.csv'
OUT=ROOT/'results/level2'; OUT.mkdir(parents=True,exist_ok=True)
if not SPLIT.exists():
    raise FileNotFoundError('Authoritative Level-2 group-aware split is required; run notebook 02 first.')


In [ ]:
from src.experiments.io import iter_jsonl
from src.experiments.level2 import support_threshold_analysis

with SPLIT.open(newline='',encoding='utf-8') as f:
    partition_by_id={r['canonical_record_id']:r['partition'] for r in csv.DictReader(f)}
parts={'train':[],'validation':[],'test':[]}
for record in iter_jsonl(DATA):
    partition=partition_by_id.get(str(record['canonical_record_id']))
    if partition not in parts:
        raise ValueError(f'Missing or invalid Level-2 partition for {record["canonical_record_id"]}')
    parts[partition].append(record)

rows=[]
for source in ['papers_with_code','bio.tools']:
    rows.extend(support_threshold_analysis(
        train_rows=parts['train'],
        validation_rows=parts['validation'],
        test_rows=parts['test'],
        source=source,
        thresholds=(5,10,20),
    ))
(OUT/'support_threshold_analysis.json').write_text(json.dumps(rows,indent=2)+'\n')
csv_rows=[]
for result in rows:
    flat={k:v for k,v in result.items() if k not in {'eligible_labels','training_support','partition_label_support'}}
    flat['eligible_labels']=json.dumps(result['eligible_labels'])
    flat['training_support']=json.dumps(result['training_support'],sort_keys=True)
    flat['partition_label_support']=json.dumps(result['partition_label_support'],sort_keys=True)
    csv_rows.append(flat)
with (OUT/'support_threshold_analysis.csv').open('w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=list(csv_rows[0])); w.writeheader(); w.writerows(csv_rows)
print(json.dumps(rows,indent=2))
